# Portrait Matting with MODNet

<div style="display:flex; flex-wrap:wrap; align-items:center;">
  <a style="margin-right:10px; margin-bottom:6px;" href="https://pepy.tech/projects/uniface"><img alt="PyPI Downloads" src="https://static.pepy.tech/personalized-badge/uniface?period=total&units=international_system&left_color=grey&right_color=blue&left_text=Downloads"></a>
  <a style="margin-right:10px; margin-bottom:6px;" href="https://pypi.org/project/uniface/"><img alt="PyPI Version" src="https://img.shields.io/pypi/v/uniface.svg"></a>
  <a style="margin-right:10px; margin-bottom:6px;" href="https://opensource.org/licenses/MIT"><img alt="License" src="https://img.shields.io/badge/License-MIT-blue.svg"></a>
  <a style="margin-bottom:6px;" href="https://github.com/yakhyo/uniface"><img alt="GitHub Stars" src="https://img.shields.io/github/stars/yakhyo/uniface.svg?style=social"></a>
</div>

**UniFace** is a lightweight, production-ready, all-in-one face analysis library built on ONNX Runtime.

🔗 **GitHub**: [github.com/yakhyo/uniface](https://github.com/yakhyo/uniface) | 📚 **Docs**: [yakhyo.github.io/uniface](https://yakhyo.github.io/uniface)

---

This notebook demonstrates portrait matting using **MODNet** — a trimap-free model that produces soft alpha mattes from full images. No face detection or cropping required.

## 1. Install UniFace

In [ ]:
%pip install -q "uniface[cpu]"

# Clone repo for assets (Colab only)
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    if not os.path.exists('uniface'):
        !git clone --depth 1 https://github.com/yakhyo/uniface.git
    os.chdir('uniface/examples')

## 2. Import Libraries

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

import uniface
from uniface.matting import MODNet

print(f"UniFace version: {uniface.__version__}")

## 3. Initialize Model

MODNet has two variants:
- **PHOTOGRAPHIC** (default): optimized for high-quality portrait photos
- **WEBCAM**: optimized for real-time webcam feeds

In [ ]:
matting = MODNet()

print(f"Input size: {matting.input_size}")
print(f"Input name: {matting.input_name}")
print(f"Output names: {matting.output_names}")

## 4. Helper Functions

In [ ]:
def compose(image, matte, background=None):
    """Composite foreground over a background using the alpha matte."""
    h, w = image.shape[:2]
    matte_3ch = matte[:, :, np.newaxis]

    if background is None:
        bg = np.full_like(image, (0, 177, 64), dtype=np.uint8)
    else:
        bg = cv2.resize(background, (w, h), interpolation=cv2.INTER_AREA)

    return (image * matte_3ch + bg * (1 - matte_3ch)).astype(np.uint8)


def show_results(image, matte):
    """Display original, matte, and green screen as a single merged image."""
    matte_vis = cv2.cvtColor((matte * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    green = compose(image, matte)
    merged = np.hstack([image, matte_vis, green])

    plt.figure(figsize=(18, 6))
    plt.imshow(cv2.cvtColor(merged, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.tight_layout()
    plt.show()

## 5. Basic Matting

In [ ]:
image = cv2.imread("../assets/demos/src_portrait1.jpg")
print(f"Image shape: {image.shape}")

matte = matting.predict(image)
print(f"Matte shape: {matte.shape}")
print(f"Matte dtype: {matte.dtype}")
print(f"Matte range: [{matte.min():.3f}, {matte.max():.3f}]")

show_results(image, matte)

## 6. Transparent Background (RGBA)

In [ ]:
alpha = (matte * 255).astype(np.uint8)
rgba = cv2.cvtColor(image, cv2.COLOR_BGR2BGRA)
rgba[:, :, 3] = alpha

# Checkerboard background to visualize transparency
h, w = image.shape[:2]
checker = np.zeros((h, w, 3), dtype=np.uint8)
block = 20
for y in range(0, h, block):
    for x in range(0, w, block):
        if (y // block + x // block) % 2 == 0:
            checker[y:y+block, x:x+block] = 200
        else:
            checker[y:y+block, x:x+block] = 255

matte_3ch = matte[:, :, np.newaxis]
rgba_vis = (image * matte_3ch + checker * (1 - matte_3ch)).astype(np.uint8)

merged = np.hstack([image, rgba_vis])

plt.figure(figsize=(16, 5))
plt.imshow(cv2.cvtColor(merged, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.tight_layout()
plt.show()

print(f"RGBA shape: {rgba.shape}")

## 7. Custom Background

In [ ]:
# Create a gradient background
h, w = image.shape[:2]
gradient = np.zeros((h, w, 3), dtype=np.uint8)
for y in range(h):
    ratio = y / h
    gradient[y, :] = [int(180 * (1 - ratio)), int(100 + 80 * ratio), int(220 * ratio)]

custom_bg = compose(image, matte, gradient)
green_bg = compose(image, matte)

merged = np.hstack([image, green_bg, custom_bg])

plt.figure(figsize=(18, 6))
plt.imshow(cv2.cvtColor(merged, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.tight_layout()
plt.show()

## Summary

MODNet provides trimap-free portrait matting:

- **`predict(image)`** — returns `(H, W)` float32 alpha matte in `[0, 1]`
- **No face detection needed** — works on full images directly
- **Two variants** — `PHOTOGRAPHIC` for photos, `WEBCAM` for real-time
- **Compositing** — use the matte for transparent PNGs, green screen, or custom backgrounds

For more details, see the [Matting docs](https://yakhyo.github.io/uniface/modules/matting/).